# Storybook Agent

Google ADK + Clean Architecture로 구현한 어린이 동화책 생성 에이전트.

| 에이전트 | 타입 | 역할 |
|---------|------|------|
| `StoryWriter` | `LlmAgent` (output_schema) | 테마 → 5페이지 구조화 스토리 생성 → `state["story_data"]` |
| `PageIllustrator` | `Agent` (tool) | state에서 현재 페이지 읽어 Imagen으로 이미지 생성 |
| `Illustrator` | `LoopAgent` (×5) | PageIllustrator를 5회 반복 실행 |
| `StorybookCreator` | `SequentialAgent` | StoryWriter → Illustrator 순차 실행 |

## 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 1. Domain Layer — 엔티티

In [2]:
from pydantic import BaseModel


class StoryPage(BaseModel):
    page_number: int
    text: str
    visual_description: str


class StoryBook(BaseModel):
    theme: str
    title: str
    pages: list[StoryPage]

## 2. Infrastructure Layer — Imagen API

In [3]:
from google import genai
from google.genai import types as genai_types


def generate_image(prompt: str) -> bytes:
    client = genai.Client()
    response = client.models.generate_images(
        model="imagen-3.0-generate-002",
        prompt=prompt,
        config=genai_types.GenerateImagesConfig(
            number_of_images=1,
            aspect_ratio="4:3",
        ),
    )
    return response.generated_images[0].image.image_bytes

## 3. Adapters Layer — ADK 에이전트 정의

### 3-1. Story Writer Agent

`LlmAgent` + `output_schema=StoryBook` → 5페이지 스토리를 구조화된 JSON으로 출력하고 `state["story_data"]`에 자동 저장.

In [4]:
from google.adk.agents import LlmAgent

STORY_WRITER_INSTRUCTION = """
You are a creative children's book author who writes in Korean.

Given a theme, write a 5-page children's story with:
- Simple, warm language suitable for young children (ages 4-8)
- A clear narrative arc: introduction → development → climax → resolution → ending
- Each page: a short story text (2-3 Korean sentences) and a vivid visual_description in English

Output exactly 5 pages in the structured format.
"""

story_writer_agent = LlmAgent(
    name="StoryWriter",
    description="테마를 받아 5페이지 분량의 어린이 동화를 구조화된 데이터로 작성합니다.",
    instruction=STORY_WRITER_INSTRUCTION,
    model="gemini-2.0-flash",
    output_schema=StoryBook,
    output_key="story_data",
)

/Users/hans/Desktop/project/movie-expert-agent/storybook-agent/.venv/lib/python3.13/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey
/Users/hans/Desktop/project/movie-expert-agent/storybook-agent/.venv/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### 3-2. Page Illustrator Agent

`state["current_page_index"]`로 현재 페이지를 추적하며, 매 호출마다 한 페이지씩 Imagen 이미지를 생성해 Artifact로 저장.

In [5]:
from google.adk.agents import Agent
from google.adk.tools.tool_context import ToolContext
from google.genai import types


async def illustrate_current_page(tool_context: ToolContext) -> dict:
    """state에서 현재 페이지를 읽어 이미지를 생성하고 Artifact로 저장합니다."""
    state = tool_context.state
    story_data = state.get("story_data")
    if not story_data:
        return {"error": "story_data not found in state"}

    page_index = state.get("current_page_index", 0)
    pages = story_data["pages"]

    if page_index >= len(pages):
        return {"status": "all pages illustrated"}

    page = pages[page_index]
    page_number = page["page_number"]
    visual_description = page["visual_description"]

    image_bytes = generate_image(
        f"Children's book illustration, soft watercolor style: {visual_description}"
    )

    artifact = types.Part(
        inline_data=types.Blob(mime_type="image/jpeg", data=image_bytes)
    )
    filename = f"page_{page_number}_image.jpeg"
    await tool_context.save_artifact(filename, artifact)

    state["current_page_index"] = page_index + 1

    return {
        "page_number": page_number,
        "text": page["text"],
        "visual_description": visual_description,
        "artifact": filename,
        "status": "illustrated",
    }


page_illustrator_agent = Agent(
    name="PageIllustrator",
    description="state에서 현재 페이지를 읽어 이미지를 생성하고 Artifact로 저장합니다.",
    instruction="illustrate_current_page 도구를 호출하여 현재 페이지의 이미지를 생성하세요. 도구를 한 번만 호출하세요.",
    model="gemini-2.0-flash",
    tools=[illustrate_current_page],
)

### 3-3. Illustrator Agent (LoopAgent)

`LoopAgent`로 `PageIllustrator`를 5회 반복 → 각 이터레이션마다 한 페이지씩 처리.

In [6]:
from google.adk.agents import LoopAgent

illustrator_agent = LoopAgent(
    name="Illustrator",
    description="5페이지 동화의 각 페이지 이미지를 순차적으로 생성합니다.",
    sub_agents=[page_illustrator_agent],
    max_iterations=5,
)

### 3-4. Root Agent (SequentialAgent)

In [7]:
from google.adk.agents import SequentialAgent

root_agent = SequentialAgent(
    name="StorybookCreator",
    description="테마를 받아 5페이지 분량의 삽화가 포함된 어린이 동화책을 만듭니다.",
    sub_agents=[story_writer_agent, illustrator_agent],
)

## 4. 실행

`InMemorySessionService` + `InMemoryArtifactService`로 세션을 관리하고 `Runner`로 에이전트를 실행합니다.

In [ ]:
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.genai import types

APP_NAME = "storybook"
USER_ID = "user_1"

session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

theme = input("동화 테마를 입력하세요 (예: 토끼가 모험을 떠나는 이야기): ").strip()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
)

runner = Runner(
    agent=root_agent,
    session_service=session_service,
    app_name=APP_NAME,
    artifact_service=artifact_service,
)

message = types.Content(
    role="user",
    parts=[types.Part(text=f"테마: {theme}")],
)

print(f"\n'{theme}' 테마로 동화책을 만들고 있습니다...\n")

async for event in runner.run_async(
    user_id=USER_ID,
    session_id=session.id,
    new_message=message,
):
    if event.is_final_response() and event.content:
        print(event.content.parts[0].text)

print("\n동화책 생성 완료!")

## 5. 결과 출력 — 스토리 텍스트 + 이미지

In [ ]:
from IPython.display import Image, display

updated_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id,
)

story_data = updated_session.state.get("story_data", {})
title = story_data.get("title", "동화책")
pages = story_data.get("pages", [])

print(f"# {title}\n")

for page in pages:
    page_num = page["page_number"]
    print(f"## Page {page_num}")
    print(f"Text: {page['text']}")
    print(f"Visual: {page['visual_description']}\n")

    try:
        artifact = await artifact_service.load_artifact(
            app_name=APP_NAME,
            user_id=USER_ID,
            session_id=session.id,
            filename=f"page_{page_num}_image.jpeg",
        )
        if artifact:
            display(Image(data=artifact.inline_data.data))
    except Exception as e:
        print(f"[이미지 로드 실패: {e}]")

    print()